In [1]:
#path setup
#since the notebook is in a subfolder, we need to add the src folder to the path
#The issue can be solved by installing the package in editable mode
from pathlib import Path
import sys
project_root = next(parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir())
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
# Import the point-level extraction and outlier functions
from ML_LC_Classifier import (
    detect_outliers_per_class,
    load_and_extract_points,
    remove_outliers,
)


In [3]:
raster_path = r"C:\AFLICM_Test\Imagery\reswir\*.tif"
final_s2 = stack_rasters(raster_path, output_path=project_root / "input_data" / "feature_stack.tif",
                         target_crs = "EPSG:32751",
                         resolution=20,
                         band_names = ["B2", "B3", "B4", "B5", "B6", "B8", "B11", "B12"])

In [ ]:
# Point-level outlier detection
RASTER_PATH = r"C:\AFLICM_Test\Imagery\reswir\merged_sentinel_2025.tif"
POINTS_PATH = r"C:\Users\AFahrezi\Documents\GitHub\Improve_pixel_based_lc_classification\input_data\GT_timor_v3_samples.geojson"
CLASS_FIELD = "ID"
CLASS_NAME_FIELD = "LULC_type"
ID_FIELD = "point_id"
REMOVE_OUTLIERS = True

point_features = load_and_extract_points(
    raster_path=RASTER_PATH,
    points_path=POINTS_PATH,
    class_field=CLASS_FIELD,
    id_field=ID_FIELD,
    class_name_field=CLASS_NAME_FIELD,
)

feature_cols = [
    column for column in point_features.columns
    if column not in {ID_FIELD, CLASS_FIELD, CLASS_NAME_FIELD}
]
flags = detect_outliers_per_class(
    df=point_features,
    feature_cols=feature_cols,
    class_col=CLASS_FIELD,
    id_col=ID_FIELD,
    class_name_col=CLASS_NAME_FIELD,
)

flags.to_csv(
    project_root / "output" / "training_point_outlier_flags_GT_Timor.csv",
    index=False,
)

# Review flags before using the cleaned table for model training.
cleaned_point_features = (
    remove_outliers(point_features, flags, id_col=ID_FIELD)
    if REMOVE_OUTLIERS
    else point_features.copy()
)
cleaned_point_features.to_csv(
    project_root / "output" / "training_points_cleaned_GT_Timor.csv",
    index=False,
)

print(f"Original samples: {len(point_features)}")
print(f"Flagged outliers: {flags['outlier'].sum()}")
print(f"Samples after filtering: {len(cleaned_point_features)}")
flags[flags["outlier"]]


,point_id,ID,outlier,score,n_in_class
453,1048,10,True,0.766125,138
108,679,3,True,0.740010,39
180,92,7,True,0.723785,19
560,1188,14,True,0.708693,41
275,836,9,True,0.705979,100
289,850,9,True,0.705012,100
457,1082,11,True,0.701502,12
473,355,12,True,0.696541,42
94,561,2,True,0.679219,22
567,44,15,True,0.679211,23
